In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ── Configuration ─────────────────────────────────────────────────────────────
ROOT = r"D:\FCAI\plain_coupon\infared_data"

SUMMARY_SUBPATH = os.path.join(
    "time_average", "average_temperature",
    "half_domain_temperature", "summary_avg_std"
)

INPUT_FILENAME  = "combined_spanwise_avg_std.csv"
OUTPUT_SUBFOLDER = "averaged_temp_distribution_plot_win1_win2"
OUTPUT_FILENAME  = "avg_temp_distribution.png"

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.labelsize":   13,
    "legend.fontsize":  11,
    "xtick.labelsize":  11,
    "ytick.labelsize":  11,
    "axes.grid":        True,
    "grid.linestyle":   "--",
    "grid.alpha":       0.4,
    "figure.dpi":       150,
})

# ── Find all case directories ─────────────────────────────────────────────────
case_dirs = sorted([
    d for d in os.listdir(ROOT)
    if os.path.isdir(os.path.join(ROOT, d)) and d.startswith("cr_")
])

print(f"Found {len(case_dirs)} case(s): {case_dirs}\n")

for case in case_dirs:
    summary_dir = os.path.join(ROOT, case, SUMMARY_SUBPATH)
    csv_path    = os.path.join(summary_dir, INPUT_FILENAME)

    if not os.path.isfile(csv_path):
        print(f"  [SKIP] {case}: combined CSV not found → {csv_path}")
        continue

    # ── Load data ──────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    x      = df["x_world_m"].values
    T_avg  = df["spanwise_avg_C"].values
    T_std  = df["spanwise_std_C"].values

    # ── Parse case label for title ─────────────────────────────────────────────
    # e.g. cr_625_phi_118  →  CR = 625 slpm, φ = 1.18
    parts = case.split("_")           # ['cr', '625', 'phi', '118']
    try:
        cr_val  = parts[1]
        phi_raw = parts[3]
        # Insert decimal point: '118' → '1.18',  '09' → '0.9'
        if len(phi_raw) == 3:
            phi_str = phi_raw[0] + "." + phi_raw[1:]
        elif len(phi_raw) == 2:
            phi_str = "0." + phi_raw
        else:
            phi_str = phi_raw
        case_title = fr"CR = {cr_val} slpm,  $\phi$ = {phi_str}"
    except IndexError:
        case_title = case

    # ── Create output directory ────────────────────────────────────────────────
    out_dir = os.path.join(summary_dir, OUTPUT_SUBFOLDER)
    os.makedirs(out_dir, exist_ok=True)

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))

    # ±1σ shaded band
    ax.fill_between(
        x * 1e3,                   # convert to mm for readability on x-axis
        T_avg - T_std,
        T_avg + T_std,
        alpha=0.25,
        color="#2196F3",
        label=r"$\pm 1\sigma$",
    )

    # Mean temperature line
    ax.plot(
        x * 1e3,
        T_avg,
        color="#0D47A1",
        linewidth=1.8,
        label=r"$\overline{T}$ (spanwise avg)",
    )

    # ── Axes formatting ────────────────────────────────────────────────────────
    ax.set_xlabel("Streamwise distance  [mm]")
    ax.set_ylabel("Temperature  [°C]")
    ax.set_title(f"Average Temperature Distribution\n{case_title}", pad=10)
    ax.legend(loc="best", framealpha=0.8)

    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.tick_params(which="minor", length=3, color="gray")

    # Add a second x-axis tick label in metres at the top
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xlabel("Streamwise distance  [m]", labelpad=6)
    xticks_mm = ax.get_xticks()
    ax2.set_xticks(xticks_mm)
    ax2.set_xticklabels([f"{v/1e3:.4f}" for v in xticks_mm], fontsize=9)
    ax2.tick_params(axis="x", labelsize=9)

    fig.tight_layout()

    # ── Save ───────────────────────────────────────────────────────────────────
    out_path = os.path.join(out_dir, OUTPUT_FILENAME)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [OK] {case}: plot saved → {out_path}")

print("\nDone.")

Found 15 case(s): ['cr_1400_phi_085', 'cr_1400_phi_09', 'cr_1400_phi_10', 'cr_1400_phi_118', 'cr_1400_phi_137', 'cr_2300_phi_085', 'cr_2300_phi_09', 'cr_2300_phi_10', 'cr_2300_phi_118', 'cr_2300_phi_137', 'cr_500_phi_085', 'cr_500_phi_09', 'cr_500_phi_10', 'cr_500_phi_118', 'cr_500_phi_137']



findfont: Font family ['cmsy10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmr10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmtt10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.


  [OK] cr_1400_phi_085: plot saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\averaged_temp_distribution_plot_win1_win2\avg_temp_distribution.png
  [OK] cr_1400_phi_09: plot saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_09\time_average\average_temperature\half_domain_temperature\summary_avg_std\averaged_temp_distribution_plot_win1_win2\avg_temp_distribution.png
  [OK] cr_1400_phi_10: plot saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_10\time_average\average_temperature\half_domain_temperature\summary_avg_std\averaged_temp_distribution_plot_win1_win2\avg_temp_distribution.png
  [OK] cr_1400_phi_118: plot saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_118\time_average\average_temperature\half_domain_temperature\summary_avg_std\averaged_temp_distribution_plot_win1_win2\avg_temp_distribution.png
  [OK] cr_1400_phi_137: plot saved → D:\FCAI\plain_coupon\infared_data\cr_1400_phi_137\time_